# Code Review Evaluation Skeleton

這個 Notebook 用來評估以 `git diff` 為輸入的 code review agent。

評估流程：
1. 從 collected PR JSONL 載入單筆資料。
2. 取出 `last_commit_gitdiff` 當作 agent input。
3. 由 agent 產生兩份輸出：
   - agent generated PR description
   - code review report
4. 將 generated PR description 與原始 `pr_description` 比較。
5. 將 generated code review report 與原始 `comments` 比較。

目前 notebook 先提供 evaluation skeleton，agent 執行先保留 command line placeholder。

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

DATASET_PATH = Path("exports/spring-projects_spring-ai_prs_all.jsonl")
SAMPLE_INDEX = 0
AGENT_COMMAND: Optional[List[str]] = None
AGENT_TIMEOUT_SECONDS = 180
PREVIEW_CHARS = 600

print(f"Dataset: {DATASET_PATH.resolve()}")
print(f"Sample index: {SAMPLE_INDEX}")
print("Agent command configured:", "yes" if AGENT_COMMAND else "no (placeholder mode)")

In [ ]:
@dataclass
class AgentOutput:
    pr_description: str
    code_review_report: str
    raw_stdout: str = ""


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def pick_sample(records: List[Dict[str, Any]], index: int) -> Dict[str, Any]:
    if not records:
        raise ValueError("Dataset is empty")
    if index < 0 or index >= len(records):
        raise IndexError(f"Sample index {index} out of range: 0..{len(records) - 1}")
    return records[index]


records = load_jsonl(DATASET_PATH)
sample = pick_sample(records, SAMPLE_INDEX)

print(f"Loaded {len(records)} records")
print("Selected PR:", sample.get("url"))
print("Title:", sample.get("title", ""))
print()
print("Git diff preview:")
print((sample.get("last_commit_gitdiff") or "")[:PREVIEW_CHARS])

In [ ]:
import shlex
import subprocess


def build_agent_payload(record: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "url": record.get("url"),
        "title": record.get("title") or "",
        "git_diff": record.get("last_commit_gitdiff") or "",
    }


def run_agent(payload: Dict[str, Any]) -> AgentOutput:
    if not AGENT_COMMAND:
        return AgentOutput(
            pr_description="PLACEHOLDER: generated PR description",
            code_review_report="PLACEHOLDER: generated code review report",
            raw_stdout="",
        )

    command = [*AGENT_COMMAND, json.dumps(payload, ensure_ascii=False)]
    print("Running:", shlex.join(command))
    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        timeout=AGENT_TIMEOUT_SECONDS,
        check=True,
    )
    stdout = completed.stdout.strip()
    response = json.loads(stdout)
    return AgentOutput(
        pr_description=response.get("pr_description") or "",
        code_review_report=response.get("code_review_report") or "",
        raw_stdout=stdout,
    )


payload = build_agent_payload(sample)
agent_output = run_agent(payload)
print(json.dumps(agent_output.__dict__, ensure_ascii=False, indent=2)[:PREVIEW_CHARS])

In [ ]:
from collections import Counter
from difflib import SequenceMatcher


def normalize_text(text: str) -> str:
    return " ".join((text or "").split()).strip().lower()


def similarity_score(left: str, right: str) -> float:
    return SequenceMatcher(None, normalize_text(left), normalize_text(right)).ratio()


def token_overlap(left: str, right: str) -> float:
    left_tokens = Counter(normalize_text(left).split())
    right_tokens = Counter(normalize_text(right).split())
    if not left_tokens or not right_tokens:
        return 0.0
    overlap = sum((left_tokens & right_tokens).values())
    base = max(sum(left_tokens.values()), sum(right_tokens.values()))
    return overlap / base if base else 0.0


def comment_texts(record: Dict[str, Any]) -> List[str]:
    texts: List[str] = []
    for comment in record.get("comments") or []:
        body = (comment.get("body") or "").strip()
        state = (comment.get("state") or "").strip()
        combined = "\n".join(part for part in [state, body] if part).strip()
        if combined:
            texts.append(combined)
    return texts


def compare_pr_description(record: Dict[str, Any], generated: AgentOutput) -> Dict[str, Any]:
    reference = record.get("pr_description") or ""
    candidate = generated.pr_description
    return {
        "reference_length": len(reference),
        "candidate_length": len(candidate),
        "similarity": round(similarity_score(reference, candidate), 4),
        "token_overlap": round(token_overlap(reference, candidate), 4),
    }


def compare_review_report(record: Dict[str, Any], generated: AgentOutput) -> Dict[str, Any]:
    reference_comments = comment_texts(record)
    reference_blob = "\n\n".join(reference_comments)
    candidate = generated.code_review_report
    return {
        "reference_comment_count": len(reference_comments),
        "candidate_length": len(candidate),
        "similarity": round(similarity_score(reference_blob, candidate), 4),
        "token_overlap": round(token_overlap(reference_blob, candidate), 4),
        "reference_preview": reference_comments[:3],
    }

In [ ]:
from dataclasses import asdict


pr_description_eval = compare_pr_description(sample, agent_output)
review_report_eval = compare_review_report(sample, agent_output)

result = {
    "sample_url": sample.get("url"),
    "title": sample.get("title") or "",
    "input_git_diff_length": len(sample.get("last_commit_gitdiff") or ""),
    "agent_output": asdict(agent_output),
    "pr_description_evaluation": pr_description_eval,
    "review_report_evaluation": review_report_eval,
}

print(json.dumps(result, ensure_ascii=False, indent=2))

# Next step:
# 1. 將 AGENT_COMMAND 換成實際 command line。
# 2. 規範 agent stdout JSON schema: {"pr_description": "...", "code_review_report": "..."}
# 3. 視需要再補 precision/recall、LLM-as-a-judge、結構化 review finding 對齊等更強的評估。